# OSM-Daten pro Bundesland herunterladen und filtern

Dieses Skript lädt OSM-PBF-Dateien für jedes deutsche Bundesland von Geofabrik herunter,
filtert alle Straßen (highways) mit Osmium,
und speichert die gefilterten PBF-Dateien.

**Vorteil:** Jedes Bundesland kann später einzeln geladen werden → sehr RAM-effizient!

**Output:** `processed_osm_files/processed_highways_DE-XX_latest.pbf` (~50-500 MB je nach Bundesland)

In [1]:
import os
import requests
import subprocess
import json
from datetime import datetime
from tqdm import tqdm

import geopandas as gpd
import pandas as pd

from pathlib import Path

from config import GEOFABRIK_CONFIG, PROCESSING_CONFIG, MAPILLARY_CONFIG, TILES_CONFIG

## Check if osmium is installed

In [2]:
try:
    result = subprocess.run(['osmium', '--version'], check=True, capture_output=True, text=True)
    print(f"Osmium version: {result.stdout.strip()}")
except subprocess.CalledProcessError as e:
    print(f"Error running Osmium: {e}")

Osmium version: osmium version 1.18.0
libosmium version 2.22.0
Supported PBF compression types: none zlib lz4

Copyright (C) 2013-2025  Jochen Topf <jochen@topf.org>
License: GNU GENERAL PUBLIC LICENSE Version 3 <https://gnu.org/licenses/gpl.html>.
This is free software: you are free to change and redistribute it.
There is NO WARRANTY, to the extent permitted by law.


## Downloading OSM data from Geofabrik

In [3]:
# Geofabrik URLs für deutsche Bundesländer
# https://download.geofabrik.de/europe/germany.html

ALL_BUNDESLAND_URLS = {
    "DE-BW": "https://download.geofabrik.de/europe/germany/baden-wuerttemberg-latest.osm.pbf",
    "DE-BY": "https://download.geofabrik.de/europe/germany/bayern-latest.osm.pbf",
    "DE-BE": "https://download.geofabrik.de/europe/germany/berlin-latest.osm.pbf",
    "DE-BB": "https://download.geofabrik.de/europe/germany/brandenburg-latest.osm.pbf",
    "DE-HB": "https://download.geofabrik.de/europe/germany/bremen-latest.osm.pbf",
    "DE-HH": "https://download.geofabrik.de/europe/germany/hamburg-latest.osm.pbf",
    "DE-HE": "https://download.geofabrik.de/europe/germany/hessen-latest.osm.pbf",
    "DE-MV": "https://download.geofabrik.de/europe/germany/mecklenburg-vorpommern-latest.osm.pbf",
    "DE-NI": "https://download.geofabrik.de/europe/germany/niedersachsen-latest.osm.pbf",
    "DE-NW": "https://download.geofabrik.de/europe/germany/nordrhein-westfalen-latest.osm.pbf",
    "DE-RP": "https://download.geofabrik.de/europe/germany/rheinland-pfalz-latest.osm.pbf",
    "DE-SL": "https://download.geofabrik.de/europe/germany/saarland-latest.osm.pbf",
    "DE-SN": "https://download.geofabrik.de/europe/germany/sachsen-latest.osm.pbf",
    "DE-ST": "https://download.geofabrik.de/europe/germany/sachsen-anhalt-latest.osm.pbf",
    "DE-SH": "https://download.geofabrik.de/europe/germany/schleswig-holstein-latest.osm.pbf",
    "DE-TH": "https://download.geofabrik.de/europe/germany/thueringen-latest.osm.pbf",
}

# Filter Bundesländer basierend auf config.py
selected_bundeslaender = GEOFABRIK_CONFIG.get("bundeslaender")
if selected_bundeslaender:
    # Nur die ausgewählten Bundesländer verarbeiten
    BUNDESLAND_URLS = {k: v for k, v in ALL_BUNDESLAND_URLS.items() if k in selected_bundeslaender}
    print(f"🎯 Ausgewählte Bundesländer: {len(BUNDESLAND_URLS)} von {len(ALL_BUNDESLAND_URLS)}")
else:
    # Alle Bundesländer verarbeiten
    BUNDESLAND_URLS = ALL_BUNDESLAND_URLS
    print(f"📍 Alle Bundesländer: {len(BUNDESLAND_URLS)}")

print(f"{'─'*70}")
for bl, url in BUNDESLAND_URLS.items():
    print(f"  {bl}: {url.split('/')[-1]}")

📍 Alle Bundesländer: 16
──────────────────────────────────────────────────────────────────────
  DE-BW: baden-wuerttemberg-latest.osm.pbf
  DE-BY: bayern-latest.osm.pbf
  DE-BE: berlin-latest.osm.pbf
  DE-BB: brandenburg-latest.osm.pbf
  DE-HB: bremen-latest.osm.pbf
  DE-HH: hamburg-latest.osm.pbf
  DE-HE: hessen-latest.osm.pbf
  DE-MV: mecklenburg-vorpommern-latest.osm.pbf
  DE-NI: niedersachsen-latest.osm.pbf
  DE-NW: nordrhein-westfalen-latest.osm.pbf
  DE-RP: rheinland-pfalz-latest.osm.pbf
  DE-SL: saarland-latest.osm.pbf
  DE-SN: sachsen-latest.osm.pbf
  DE-ST: sachsen-anhalt-latest.osm.pbf
  DE-SH: schleswig-holstein-latest.osm.pbf
  DE-TH: thueringen-latest.osm.pbf


In [4]:
def should_update_osm_data(bundesland_code, max_age_days=2):
    """
    Prüft ob OSM-Daten für ein Bundesland aktualisiert werden müssen.
    
    Args:
        bundesland_code: Der Bundesland-Code (z.B. "DE-BE")
        max_age_days: Maximales Alter in Tagen (default: 2)
    
    Returns:
        True wenn Update nötig, False sonst
    """
    if not os.path.exists("osm_metadata.json"):
        return True  # Keine Metadata -> Download nötig
    
    try:
        with open("osm_metadata.json", "r") as f:
            metadata = json.load(f)
        
        bundeslaender = metadata.get("bundeslaender", {})
        if bundesland_code not in bundeslaender:
            return True  # Bundesland nicht in Metadata -> Download nötig
        
        # Parse Timestamp
        timestamp_str = bundeslaender[bundesland_code]
        timestamp = datetime.fromisoformat(timestamp_str.replace('Z', '+00:00'))
        
        # Berechne Alter
        now = datetime.now(timestamp.tzinfo)
        age_days = (now - timestamp).days
        
        if age_days >= max_age_days:
            print(f"  ℹ️  OSM-Daten für {bundesland_code} sind {age_days} Tage alt → Update nötig")
            return True
        else:
            print(f"  ℹ️  OSM-Daten für {bundesland_code} sind {age_days} Tage alt → aktuell genug")
            return False
            
    except Exception as e:
        print(f"  ⚠️  Fehler beim Lesen der Metadata: {e} → Download wird durchgeführt")
        return True


def download_bundesland_pbf(bundesland_code, url, force_download=False):
    """Download PBF file for a specific Bundesland."""
    folder_download = GEOFABRIK_CONFIG["download_folder"]
    os.makedirs(folder_download, exist_ok=True)
    
    filename = f"{bundesland_code}_{url.split('/')[-1]}"
    file_path = os.path.join(folder_download, filename)
    
    # Prüfe ob Download nötig ist (außer wenn force_download=True)
    if os.path.exists(file_path) and not force_download:
        print(f"  ✓ Bereits vorhanden: {filename}")
        return file_path
    
    if os.path.exists(file_path):
        print(f"  🔄 Überschreibe vorhandene Datei: {filename}")
        os.remove(file_path)
    
    print(f"  📥 Lade herunter: {filename}")
    try:
        response = requests.get(url, stream=True, timeout=300, allow_redirects=True)
        response.raise_for_status()
        
        # Get file size for progress bar
        total_size = int(response.headers.get('content-length', 0))
        
        with open(file_path, "wb") as f, tqdm(
            total=total_size,
            unit='B',
            unit_scale=True,
            desc=f"    {bundesland_code}",
            leave=False
        ) as pbar:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
                pbar.update(len(chunk))
        
        print(f"  ✓ Download abgeschlossen: {filename}")
        return file_path
        
    except Exception as e:
        print(f"  ❌ Fehler beim Download von {bundesland_code}: {e}")
        if os.path.exists(file_path):
            os.remove(file_path)
        return None


def filter_highways_osmium(input_pbf, output_pbf, bundesland_code, force_filter=False):
    """Filter highways from PBF using Osmium."""
    if os.path.exists(output_pbf) and not force_filter:
        print(f"  ✓ Bereits gefiltert: {os.path.basename(output_pbf)}")
        return True
    
    if os.path.exists(output_pbf):
        print(f"  🔄 Überschreibe vorhandene gefilterte Datei")
        os.remove(output_pbf)
    
    print(f"  🔧 Filtere Highways für {bundesland_code}...")
    
    try:
        filter_command = [
            "osmium", "tags-filter",
            input_pbf,
            "w/highway",
            "-o", output_pbf
        ]
        subprocess.run(filter_command, check=True, capture_output=True)
        print(f"  ✓ Filterung abgeschlossen: {os.path.basename(output_pbf)}")
        return True
        
    except subprocess.CalledProcessError as e:
        print(f"  ❌ Fehler bei Osmium-Filterung für {bundesland_code}: {e}")
        if os.path.exists(output_pbf):
            os.remove(output_pbf)
        return False


def get_osm_timestamp(pbf_file):
    """Extract timestamp from PBF file using osmium."""
    try:
        result = subprocess.run([
            'osmium', 'fileinfo', pbf_file, '-g', 'header.option.timestamp'
        ], capture_output=True, text=True, check=True)
        return result.stdout.strip()
    except subprocess.CalledProcessError as e:
        return None

In [5]:
# Hauptprozess: Download und Filterung für alle Bundesländer

folder_download = GEOFABRIK_CONFIG["download_folder"]
folder_processed = GEOFABRIK_CONFIG["processed_folder"]
os.makedirs(folder_download, exist_ok=True)
os.makedirs(folder_processed, exist_ok=True)

print(f"\n{'='*70}")
print(f"Download und Filterung OSM-Daten für {len(BUNDESLAND_URLS)} Bundesländer")
print(f"{'='*70}\n")

successful = []
failed = []
timestamps = {}

for bundesland_code, url in BUNDESLAND_URLS.items():
    print(f"{'─'*70}")
    print(f"Bundesland: {bundesland_code}")
    print(f"{'─'*70}")
    
    try:
        # Prüfe ob Update nötig ist
        needs_update = should_update_osm_data(bundesland_code, max_age_days=2)
        
        # 1. Download (nur wenn nötig oder Datei fehlt)
        downloaded_pbf = download_bundesland_pbf(bundesland_code, url, force_download=needs_update)
        if not downloaded_pbf:
            failed.append(bundesland_code)
            continue
        
        # 2. Filter highways (nur wenn nötig oder Datei fehlt)
        output_pbf = os.path.join(
            folder_processed,
            f"processed_highways_{bundesland_code}_latest.pbf"
        )
        
        success = filter_highways_osmium(downloaded_pbf, output_pbf, bundesland_code, force_filter=needs_update)
        
        if success:
            # 3. Extract timestamp
            timestamp = get_osm_timestamp(output_pbf)
            if timestamp:
                timestamps[bundesland_code] = timestamp
                print(f"  📅 OSM-Daten vom: {timestamp}")
            
            successful.append(bundesland_code)
            print(f"  ✅ {bundesland_code} erfolgreich verarbeitet!\n")
        else:
            failed.append(bundesland_code)
            
    except Exception as e:
        print(f"  ❌ Fehler bei {bundesland_code}: {e}\n")
        failed.append(bundesland_code)

# Zusammenfassung
print(f"\n{'='*70}")
print(f"ZUSAMMENFASSUNG")
print(f"{'='*70}")
print(f"✅ Erfolgreich: {len(successful)}/{len(BUNDESLAND_URLS)}")
if successful:
    print(f"   {', '.join(successful)}")

if failed:
    print(f"\n❌ Fehlgeschlagen: {len(failed)}/{len(BUNDESLAND_URLS)}")
    print(f"   {', '.join(failed)}")

# Speichere Metadata
if timestamps:
    oldest_date = min(timestamps.values())
    metadata = {
        "osm_data_from": oldest_date,
        "bundeslaender": timestamps,
        "processed_date": datetime.now().isoformat()
    }
    
    with open("osm_metadata.json", "w") as f:
        json.dump(metadata, f, indent=2)
    
    print(f"\n📅 Ältestes OSM-Datum: {oldest_date}")
    print(f"💾 Metadata gespeichert: osm_metadata.json")

print(f"\n{'='*70}")
print(f"✅ FERTIG!")
print(f"{'='*70}")


Download und Filterung OSM-Daten für 16 Bundesländer

──────────────────────────────────────────────────────────────────────
Bundesland: DE-BW
──────────────────────────────────────────────────────────────────────
  🔄 Überschreibe vorhandene Datei: DE-BW_baden-wuerttemberg-latest.osm.pbf
  📥 Lade herunter: DE-BW_baden-wuerttemberg-latest.osm.pbf


  ✓ Download abgeschlossen: DE-BW_baden-wuerttemberg-latest.osm.pbf
  🔄 Überschreibe vorhandene gefilterte Datei
  🔧 Filtere Highways für DE-BW...


  ✓ Filterung abgeschlossen: processed_highways_DE-BW_latest.pbf
  📅 OSM-Daten vom: 2025-12-05T21:21:03Z
  ✅ DE-BW erfolgreich verarbeitet!

──────────────────────────────────────────────────────────────────────
Bundesland: DE-BY
──────────────────────────────────────────────────────────────────────
  🔄 Überschreibe vorhandene Datei: DE-BY_bayern-latest.osm.pbf
  📥 Lade herunter: DE-BY_bayern-latest.osm.pbf


  ✓ Download abgeschlossen: DE-BY_bayern-latest.osm.pbf
  🔄 Überschreibe vorhandene gefilterte Datei
  🔧 Filtere Highways für DE-BY...


  ✓ Filterung abgeschlossen: processed_highways_DE-BY_latest.pbf
  📅 OSM-Daten vom: 2025-12-05T21:21:03Z
  ✅ DE-BY erfolgreich verarbeitet!

──────────────────────────────────────────────────────────────────────
Bundesland: DE-BE
──────────────────────────────────────────────────────────────────────
  🔄 Überschreibe vorhandene Datei: DE-BE_berlin-latest.osm.pbf
  📥 Lade herunter: DE-BE_berlin-latest.osm.pbf


...

  ✓ Filterung abgeschlossen: processed_highways_DE-ST_latest.pbf
  📅 OSM-Daten vom: 2025-12-05T21:21:03Z
  ✅ DE-ST erfolgreich verarbeitet!

──────────────────────────────────────────────────────────────────────
Bundesland: DE-SH
──────────────────────────────────────────────────────────────────────
  🔄 Überschreibe vorhandene Datei: DE-SH_schleswig-holstein-latest.osm.pbf
  📥 Lade herunter: DE-SH_schleswig-holstein-latest.osm.pbf


  ✓ Download abgeschlossen: DE-SH_schleswig-holstein-latest.osm.pbf
  🔄 Überschreibe vorhandene gefilterte Datei
  🔧 Filtere Highways für DE-SH...


  ✓ Filterung abgeschlossen: processed_highways_DE-SH_latest.pbf
  📅 OSM-Daten vom: 2025-12-05T21:21:03Z
  ✅ DE-SH erfolgreich verarbeitet!

──────────────────────────────────────────────────────────────────────
Bundesland: DE-TH
──────────────────────────────────────────────────────────────────────
  🔄 Überschreibe vorhandene Datei: DE-TH_thueringen-latest.osm.pbf
  📥 Lade herunter: DE-TH_thueringen-latest.osm.pbf


  ✓ Download abgeschlossen: DE-TH_thueringen-latest.osm.pbf
  🔄 Überschreibe vorhandene gefilterte Datei
  🔧 Filtere Highways für DE-TH...


  ✓ Filterung abgeschlossen: processed_highways_DE-TH_latest.pbf
  📅 OSM-Daten vom: 2025-12-05T21:21:03Z
  ✅ DE-TH erfolgreich verarbeitet!


ZUSAMMENFASSUNG
✅ Erfolgreich: 16/16
   DE-BW, DE-BY, DE-BE, DE-BB, DE-HB, DE-HH, DE-HE, DE-MV, DE-NI, DE-NW, DE-RP, DE-SL, DE-SN, DE-ST, DE-SH, DE-TH

📅 Ältestes OSM-Datum: 2025-12-04T21:21:15Z
💾 Metadata gespeichert: osm_metadata.json

✅ FERTIG!


In [6]:
# Liste der verfügbaren gefilterten PBF-Dateien
import glob

pbf_files = glob.glob(f"{folder_processed}/processed_highways_DE-*_latest.pbf")

print(f"\n{'='*70}")
print(f"Verfügbare gefilterte PBF-Dateien: {len(pbf_files)}")
print(f"{'='*70}\n")

for pbf in sorted(pbf_files):
    size_mb = os.path.getsize(pbf) / (1024 * 1024)
    basename = os.path.basename(pbf)
    print(f"  {basename:50s} ({size_mb:6.1f} MB)")

# Zeige Metadata
if os.path.exists("osm_metadata.json"):
    with open("osm_metadata.json", "r") as f:
        metadata = json.load(f)
    
    print(f"\n📅 OSM-Daten vom: {metadata.get('osm_data_from', 'Unknown')}")
    print(f"🕒 Verarbeitet am: {metadata.get('processed_date', 'Unknown')}")
else:
    print("\n⚠️  Keine Metadata gefunden")


Verfügbare gefilterte PBF-Dateien: 16

  processed_highways_DE-BB_latest.pbf                (  65.7 MB)
  processed_highways_DE-BE_latest.pbf                (  23.6 MB)
  processed_highways_DE-BW_latest.pbf                ( 168.0 MB)
  processed_highways_DE-BY_latest.pbf                ( 225.3 MB)
  processed_highways_DE-HB_latest.pbf                (   4.3 MB)
  processed_highways_DE-HE_latest.pbf                (  85.2 MB)
  processed_highways_DE-HH_latest.pbf                (  11.7 MB)
  processed_highways_DE-MV_latest.pbf                (  24.4 MB)
  processed_highways_DE-NI_latest.pbf                ( 106.2 MB)
  processed_highways_DE-NW_latest.pbf                ( 184.0 MB)
  processed_highways_DE-RP_latest.pbf                (  71.6 MB)
  processed_highways_DE-SH_latest.pbf                (  33.2 MB)
  processed_highways_DE-SL_latest.pbf                (  11.6 MB)
  processed_highways_DE-SN_latest.pbf                (  63.8 MB)
  processed_highways_DE-ST_latest.pbf             